In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("avant :", tokenized_datasets["train"].column_names)
print("type avant :", type(tokenized_datasets["train"][0]["input_ids"]))

tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

print("après :", tokenized_datasets["train"].column_names)
print("type après :", type(tokenized_datasets["train"][0]["input_ids"]))

avant : ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']
type avant : <class 'list'>
après : ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
type après : <class 'torch.Tensor'>


In [4]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)
print("batches d'entraînement :", len(train_dataloader))
print("batches de validation :", len(eval_dataloader))

batches d'entraînement : 459
batches de validation : 51


In [7]:
for batch in train_dataloader:
    break
print({k: v.shape for k, v in batch.items()})

{'labels': torch.Size([8]), 'input_ids': torch.Size([8, 82]), 'token_type_ids': torch.Size([8, 82]), 'attention_mask': torch.Size([8, 82])}


In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
print("mode entraînement ?", model.training)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33237.75it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

mode entraînement ? False


In [9]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6997, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


In [10]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
print("réglages AdamW :", {k: optimizer.defaults[k] for k in ["lr", "betas", "eps", "weight_decay"]})

réglages AdamW : {'lr': 5e-05, 'betas': (0.9, 0.999), 'eps': 1e-08, 'weight_decay': 0.01}


In [11]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print("pas d'entraînement :", num_training_steps)
print("learning rate au départ :", lr_scheduler.get_last_lr())

pas d'entraînement : 1377
learning rate au départ : [5e-05]


In [14]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)
print("device :", device)

device : mps


In [15]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
        total_loss += loss.item()
    print(f"epoch {epoch + 1} : loss moyenne {total_loss / len(train_dataloader):.3f}")

 33%|███▎      | 460/1377 [00:50<01:44,  8.73it/s]

epoch 1 : loss moyenne 0.541


 67%|██████▋   | 920/1377 [01:40<00:48,  9.37it/s]

epoch 2 : loss moyenne 0.290


100%|██████████| 1377/1377 [02:30<00:00,  9.62it/s]

epoch 3 : loss moyenne 0.109


In [16]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
total_loss = 0
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    total_loss += outputs.loss.item()
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

print("loss de validation :", total_loss / len(eval_dataloader))
print(metric.compute())

loss de validation : 0.45522800977250527
{'accuracy': 0.8357843137254902, 'f1': 0.8854700854700854}
